# 24 — RF-DETR Small no Home · Kaggle

**Importe este arquivo como um novo notebook no Kaggle. Ative GPU T4, Internet ON e o Secret já existente `GDRIVE_RCLONE_CONFIG_B64`. Depois use “Run All”.**
Não há campos para preencher nem necessidade de preparar a base novamente.

O experimento reutiliza o Home do Drive, compara **C10** e **21/época 21**, treina RF-DETR Small a partir do COCO e gera um único pacote para análise.
Mantém **12 classes como desfecho principal** e as **seis classes da política V2 como desfecho secundário**, no mesmo limiar principal.

**Meta:** precisão ≥80% **e** recall ≥70%, simultaneamente, micro por objeto com IoU ≥0,50.
Checkpoint escolhido pelo mAP50–95 do VAL; confiança escolhida pelo nosso fitness no VAL.

**O TEST Home é histórico e já foi consultado em fases anteriores.** Nesta rodada só é aberto após a decisão ser registrada no Drive. Seu resultado não será chamado de teste novo e independente.

O orçamento é de até **140 minutos de treinamento**, incluindo validação e persistência, ajustado ao alvo de **180 minutos da sessão de inferência/treino**. O corte ocorre ao completar uma época. Instalação, autenticação e preparação dos dados vêm antes dessa contagem. As três primeiras épocas medem o tempo real; **não há promessa de 40–60 épocas nem de atingir a meta**.


## 1. Instalar as versões fixadas e conferir a GPU

No Kaggle, selecione **Settings → Accelerator → GPU T4 ×2** (ou a GPU disponível), **Internet → ON**.
Em **Add-ons → Secrets**, habilite para este notebook o Secret **`GDRIVE_RCLONE_CONFIG_B64`**, já usado no 21.
Então use **Run All**. Não é necessário anexar datasets, montar o Drive ou editar caminhos.

Uma única GPU é usada para manter o batch efetivo e o protocolo iguais; T4 ×2 não significa uso das duas GPUs.
O PyTorch CUDA fornecido pelo Kaggle é preservado. A instalação pode levar alguns minutos.


In [ ]:
import os, sys, subprocess
os.environ['WANDB_MODE']='disabled'
os.environ['WANDB_DISABLED']='true'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['YOLO_AUTOINSTALL']='false'
os.environ['YOLO_CONFIG_DIR']='/tmp/tcc24/yolo_config'
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'rfdetr[train]==1.11.2','pytorch-lightning==2.6.6',
    'transformers==5.20.1','supervision==0.31.0','ultralytics==8.4.150'])
print('Dependências instaladas. Próxima etapa: carregar o código e conferir a GPU.')


In [ ]:
import base64, gzip, hashlib, json, shutil, time
from pathlib import Path
BUNDLE = '...'
BUNDLE_SHA256 = '661276868447b50b4b35182bfd05733fa348309827fbdf97e87ff31e28415b4f'
_raw=gzip.decompress(base64.b64decode(BUNDLE))
if hashlib.sha256(_raw).hexdigest()!=BUNDLE_SHA256:
    raise ValueError('O código embutido do notebook foi alterado.')
LOCAL=Path('/tmp/tcc24');CODIGO=LOCAL/'codigo';CODIGO.mkdir(parents=True,exist_ok=True)
_sources=json.loads(_raw)
for _name,_text in _sources.items():
    _path=CODIGO/_name;_path.parent.mkdir(parents=True,exist_ok=True);_path.write_text(_text,encoding='utf-8')
for _module in list(sys.modules):
    if _module.startswith('home24_') or _module=='src' or _module.startswith('src.'):
        sys.modules.pop(_module,None)
sys.path.insert(0,str(CODIGO))
import torch, importlib.metadata
import home24_common as core
import home24_workflow as fluxo
import home24_training as treino
import home24_report as relatorio
if importlib.metadata.version('rfdetr')!='1.11.2':
    raise RuntimeError('O runtime importou outra versão do RF-DETR. Reinicie a sessão e execute este notebook desde o início.')
if not torch.cuda.is_available():
    raise RuntimeError('GPU não disponível. Selecione GPU em Settings → Accelerator no Kaggle e execute novamente.')
if shutil.disk_usage('/kaggle/working').free<15*1024**3:
    raise RuntimeError('O runtime precisa de 15 GiB livres para dados, pesos e dois estados completos de retomada.')
print('GPU:',torch.cuda.get_device_name(0))
print('RF-DETR:',importlib.metadata.version('rfdetr'),'| PyTorch:',torch.__version__)
SAIDAS=Path('/kaggle/working/resultados_tcc/notebook24');SAIDAS.mkdir(parents=True,exist_ok=True)
os.chdir('/kaggle/working')
JOB={'local':LOCAL};DRIVE=None
def etapa(nome,func):
    try:
        print('\nETAPA:',nome,flush=True)
        return func()
    except Exception as exc:
        _diagnostico=fluxo.failure(JOB,exc,nome,DRIVE)
        _destino=Path('/kaggle/working/pacote_diagnostico_24.zip')
        shutil.copyfile(_diagnostico,_destino)
        print('Diagnóstico em Output:',_destino,flush=True)
        raise


## 2. Conectar ao Drive com o Secret do Kaggle

Reutiliza `gdrive_tcc:TCC_Visao_Computacional`. Confere leitura da base e uma gravação pequena antes do treino.
A credencial fica em uma pasta privada em `/tmp`, fora de `/kaggle/working`, dos relatórios e do pacote final.
Se o Secret não estiver habilitado, a mensagem informa exatamente o que ativar, sem exibir seu conteúdo.


In [ ]:
from home24_kaggle import KaggleStore
DRIVE=etapa('2-conexao-drive',KaggleStore.from_kaggle)


## 3. Preparar Home e registrar o protocolo

Confere o ZIP histórico e suas 12 classes. Exclui os quatro arquivos em quarentena e as duas exclusões adicionais registradas no 21; remove eventuais coincidências de treino com VAL/TEST por hash e semelhança com VAL.
A contagem esperada começa em **1.818 TRAIN**, **456 VAL** e **403 TEST histórico**; exclusões adicionais, se necessárias, ficam documentadas. Os pixels e rótulos de TEST não são abertos para selecionar ou treinar.
Não há nova rotulagem nem seleção por CLIP. O treino usa somente Home, sem Open Images.


In [ ]:
JOB=etapa('3-preparacao-e-protocolo',lambda:fluxo.start(DRIVE,LOCAL,BUNDLE_SHA256,output_root=SAIDAS))
print('Execução:',JOB['context'])
print('Resultados no Drive:',DRIVE.location(JOB['remote']))


## 4. Medir as referências no mesmo Home VAL

Avalia **C10** e o checkpoint **21/época 21** preservado no último pacote recebido. Escolhe a confiança de cada configuração pela mesma grade e fitness.
As resoluções originais são mantidas: C10 640, 21 768; RF-DETR usará 512. A comparação representa os pipelines completos, sem atribuir toda diferença somente à arquitetura.
As previsões são guardadas em blocos para reutilização em uma retomada.


In [ ]:
REFERENCIAS=etapa('4-referencias-home-val',lambda:fluxo.reference_validation(JOB,DRIVE))
for nome,item in REFERENCIAS.items():
    p=item['primary']
    print(nome,'| confiança',p['threshold'],'| P/R/F1',p['detector'])


## 5. Treinar RF-DETR e medir o tempo real

O RF-DETR Small inicia no pré-treino público COCO. Uma prova automática de memória limita o batch a 1, 2 ou 4 e mantém batch efetivo 16 por acumulação. Usa FP16 na T4, EMA e validação a cada época.

**As três primeiras épocas fazem parte do treinamento principal.** Seus tempos incluem validação e gravação no Drive, permitindo estimar quantas épocas cabem no orçamento. O agendamento permanece fixado no protocolo; a estimativa não reinicia o treino.

Dois slots de retomada guardam modelo, otimizador, agendador, EMA e estado da parada antecipada. Uma publicação incompleta não substitui o último slot válido. Os arquivos são grandes; o tempo de persistência aparece na medição.

Se a sessão do Kaggle interromper, abra **este mesmo notebook** e use **Run All**. O estado compatível será recuperado automaticamente. Uma nova execução não recebe outro orçamento completo de treinamento.


In [ ]:
EXECUCAO=etapa('5-treino-rfdetr',lambda:treino.train(JOB,DRIVE))
print('Treinamento encerrado:',EXECUCAO)
if (JOB['output']/'prova_tempo_24.json').is_file():
    print('Tempo medido:',core.read(JOB['output']/'prova_tempo_24.json'))


## 6. Fixar as decisões e avaliar o TEST Home histórico

Recarrega o melhor checkpoint EMA exportado, avalia VAL e escolhe a confiança. Registra no Drive os pesos e limiares de RF-DETR, C10 e 21 **antes** de abrir o TEST nesta rodada.

Avalia as 403 imagens históricas, as 12 classes, as seis classes da V2 no mesmo limiar e os alertas. Calcula 2.000 réplicas de bootstrap pareado por imagem. Não há agrupamento validado de cenas Home; essa limitação acompanha os intervalos.

As previsões TEST e a decisão são reutilizadas em reexecuções compatíveis. O resultado continuará identificado como **TEST histórico já exposto**, mesmo que a meta seja atingida.

**Correção de avaliação:** a saída interna de background do RF-DETR é tratada antes de converter os índices para as 12 classes. A correção é registrada no pacote, preservando a identidade do treino e os blocos válidos de cache.


In [ ]:
"""Substitua a célula de CÓDIGO da etapa 6 por todo o conteúdo deste arquivo.
Execute-a na sessão atual e, ao concluir, siga para as etapas 7 e 8.
Não executa treino. Mantém o contexto, os pesos e os limiares definidos no VAL.
"""
import numpy as np

_necessarios = ['core', 'fluxo', 'JOB', 'DRIVE', 'EXECUCAO', 'etapa']
_faltantes = [nome for nome in _necessarios if nome not in globals()]
if _faltantes:
    raise RuntimeError('A sessão perdeu a inicialização: ' + ', '.join(_faltantes)
                       + '. Execute as etapas iniciais do mesmo notebook para recuperar o estado salvo.')

_RF24_AUDITORIA = {
    'correcao': '24-background-v1', 'context': JOB['context'],
    'motivo': 'RF-DETR 1.11.2 pode retornar a saída num_classes como __background__.',
    'regra': 'Somente ID 12 confirmado como __background__ é removido; IDs 0..11 são preservados.',
    'alterou_pesos': False, 'alterou_classes': False, 'treinamento_executado_pela_correcao': False,
    'inferencias_nesta_aplicacao': []
}

def _rf24_predict_corrigido(model, rows, batch_size=4):
    n = len(core.NAMES)
    if n != 12 or not isinstance(batch_size, int) or batch_size < 1:
        raise ValueError('Esta correção exige as 12 classes do notebook 24 e batch positivo.')
    contexto = getattr(model, 'model', None)
    configuracoes = [getattr(model, 'model_config', None), getattr(contexto, 'args', None)]
    contagens = [getattr(cfg, 'num_classes', None) for cfg in configuracoes]
    contagens = [v for v in contagens if v is not None]
    if not contagens or any(v != n for v in contagens):
        raise ValueError('O RF-DETR não declarou exatamente 12 classes: ' + str(contagens))
    nomes = getattr(contexto, 'class_names', None)
    if isinstance(nomes, dict):
        if set(nomes) != set(range(n)):raise ValueError('Índices dos nomes do RF-DETR incompatíveis.')
        nomes = [nomes[i] for i in range(n)]
    if nomes is None or list(nomes) != core.NAMES:
        raise ValueError('A ordem das 12 classes do RF-DETR não corresponde ao projeto.')

    cache = {}
    for start in range(0, len(rows), batch_size):
        bloco = rows[start:start + batch_size]
        saidas = model.predict([str(r['image']) for r in bloco], threshold=.01)
        if not isinstance(saidas, list):saidas = [saidas]
        if len(saidas) != len(bloco):raise ValueError('RF-DETR retornou quantidade incorreta de imagens.')
        for r, out in zip(bloco, saidas):
            ids = np.asarray(out.class_id)
            caixas = np.asarray(out.xyxy)
            confiancas = np.asarray(out.confidence)
            if ids.ndim != 1 or ids.dtype.kind not in 'iu' or caixas.shape != (len(ids), 4) or confiancas.shape != (len(ids),):
                raise ValueError('Formato das previsões RF-DETR incompatível em ' + r['id'])
            invalidos = ids[(ids < 0) | (ids > n)]
            if len(invalidos):
                raise ValueError('RF-DETR retornou IDs inesperados: ' + str(sorted(set(map(int, invalidos))))
                                 + '. A correção não descarta classes desconhecidas.')
            fundo = ids == n
            if fundo.any():
                nomes_saida = getattr(out, 'data', {}).get('class_name')
                if nomes_saida is None or np.asarray(nomes_saida).shape != ids.shape or not np.all(np.asarray(nomes_saida)[fundo] == '__background__'):
                    raise ValueError('O ID 12 não foi confirmado como __background__ pelo RF-DETR em ' + r['id'])
            valores = [
                {'classe_id': int(cid), 'classe': core.NAMES[int(cid)],
                 'bbox': [float(x) for x in caixa], 'confianca': float(conf)}
                for cid, caixa, conf in zip(ids[~fundo], caixas[~fundo], confiancas[~fundo])
            ]
            fluxo.ev.validate_pred(valores, r)
            cache[r['id']] = valores
            _RF24_AUDITORIA['inferencias_nesta_aplicacao'].append({
                'id': r['id'], 'role': r['role'], 'previsoes_nativas': len(ids),
                'background_removido': int(fundo.sum()), 'previsoes_12_classes': len(valores)})
        core.save(JOB['output'] / 'correcao_background_rf24.json', _RF24_AUDITORIA)
    return cache


_RF24_FONTE_APLICADA = "def _rf24_predict_corrigido(model, rows, batch_size=4):\n    n = len(core.NAMES)\n    if n != 12 or not isinstance(batch_size, int) or batch_size < 1:\n        raise ValueError('Esta correção exige as 12 classes do notebook 24 e batch positivo.')\n    contexto = getattr(model, 'model', None)\n    configuracoes = [getattr(model, 'model_config', None), getattr(contexto, 'args', None)]\n    contagens = [getattr(cfg, 'num_classes', None) for cfg in configuracoes]\n    contagens = [v for v in contagens if v is not None]\n    if not contagens or any(v != n for v in contagens):\n        raise ValueError('O RF-DETR não declarou exatamente 12 classes: ' + str(contagens))\n    nomes = getattr(contexto, 'class_names', None)\n    if isinstance(nomes, dict):\n        if set(nomes) != set(range(n)):raise ValueError('Índices dos nomes do RF-DETR incompatíveis.')\n        nomes = [nomes[i] for i in range(n)]\n    if nomes is None or list(nomes) != core.NAMES:\n        raise ValueError('A ordem das 12 classes do RF-DETR não corresponde ao projeto.')\n\n    cache = {}\n    for start in range(0, len(rows), batch_size):\n        bloco = rows[start:start + batch_size]\n        saidas = model.predict([str(r['image']) for r in bloco], threshold=.01)\n        if not isinstance(saidas, list):saidas = [saidas]\n        if len(saidas) != len(bloco):raise ValueError('RF-DETR retornou quantidade incorreta de imagens.')\n        for r, out in zip(bloco, saidas):\n            ids = np.asarray(out.class_id)\n            caixas = np.asarray(out.xyxy)\n            confiancas = np.asarray(out.confidence)\n            if ids.ndim != 1 or ids.dtype.kind not in 'iu' or caixas.shape != (len(ids), 4) or confiancas.shape != (len(ids),):\n                raise ValueError('Formato das previsões RF-DETR incompatível em ' + r['id'])\n            invalidos = ids[(ids < 0) | (ids > n)]\n            if len(invalidos):\n                raise ValueError('RF-DETR retornou IDs inesperados: ' + str(sorted(set(map(int, invalidos))))\n                                 + '. A correção não descarta classes desconhecidas.')\n            fundo = ids == n\n            if fundo.any():\n                nomes_saida = getattr(out, 'data', {}).get('class_name')\n                if nomes_saida is None or np.asarray(nomes_saida).shape != ids.shape or not np.all(np.asarray(nomes_saida)[fundo] == '__background__'):\n                    raise ValueError('O ID 12 não foi confirmado como __background__ pelo RF-DETR em ' + r['id'])\n            valores = [\n                {'classe_id': int(cid), 'classe': core.NAMES[int(cid)],\n                 'bbox': [float(x) for x in caixa], 'confianca': float(conf)}\n                for cid, caixa, conf in zip(ids[~fundo], caixas[~fundo], confiancas[~fundo])\n            ]\n            fluxo.ev.validate_pred(valores, r)\n            cache[r['id']] = valores\n            _RF24_AUDITORIA['inferencias_nesta_aplicacao'].append({\n                'id': r['id'], 'role': r['role'], 'previsoes_nativas': len(ids),\n                'background_removido': int(fundo.sum()), 'previsoes_12_classes': len(valores)})\n        core.save(JOB['output'] / 'correcao_background_rf24.json', _RF24_AUDITORIA)\n    return cache\n"
_RF24_DESTINO = JOB['output'] / 'codigo' / 'correcao_background_rf24.py'
_RF24_DESTINO.parent.mkdir(parents=True, exist_ok=True)
_RF24_DESTINO.write_text(_RF24_FONTE_APLICADA, encoding='utf-8')
_RF24_AUDITORIA['sha256_funcao_aplicada'] = 'a9c35e0da436783e9bb2ea00d83efbd3235cf2fef0b0b276609c3e7122d620c1'

fluxo.ev.predict_rf = _rf24_predict_corrigido
core.save(JOB['output'] / 'correcao_background_rf24.json', _RF24_AUDITORIA)
print('Correção aplicada: saída de background tratada; pesos e 12 classes preservados.')
RESULTADO = etapa('6-selecao-congelamento-test-historico', lambda: fluxo.evaluate(JOB, DRIVE, EXECUCAO))
for nome, p in RESULTADO['test_historico'].items():
    print(nome, '| P/R/F1 no TEST histórico:', p['detector'])


## 7. Diagnóstico externo NYU, quando disponível

Procura automaticamente a preparação 22 e a decisão congelada do 23. Se os arquivos estiverem disponíveis e houver tempo reservado, avalia RF-DETR nas seis classes comuns do NYU, com o peso e a confiança já fixados. Exclui coincidências de pixels ou candidatos pHash com Home antes da inferência externa.

Se a preparação não for localizada ou o orçamento estiver no fim, registra a omissão e continua para o pacote principal. Não exige mover arquivos nem preencher caminhos. O diagnóstico não é comparação direta com o 23 após exclusões e não garante independência entre fontes.


In [ ]:
EXTERNO=etapa('7-diagnostico-nyu-opcional',lambda:fluxo.external_nyu(JOB,DRIVE,RESULTADO))
print('NYU executado:',EXTERNO['executed'])
if not EXTERNO['executed']:print(EXTERNO.get('reason'))


## 8. Gerar o relatório e disponibilizar o pacote no Kaggle

O arquivo a enviar para análise é **`pacote_experimento_24.zip`**. Ele contém o melhor peso de inferência RF-DETR, código, protocolo, resultados, previsões e intervalos. Os dois estados completos de retomada permanecem no Drive e não entram nesse ZIP.

Se houver erro em alguma etapa, procure o **`pacote_diagnostico_24.zip`** criado na pasta informada. Ele pode conter resultados parciais; sua existência não indica treinamento concluído.

Ao terminar, baixe **`pacote_experimento_24.zip`** pela seção **Output / Files → `/kaggle/working`** ou pelo link exibido. Há também uma cópia automática no Drive.


In [ ]:
from IPython.display import display,HTML,FileLink
PAGINA=etapa('8-relatorio',lambda:relatorio.build(JOB,RESULTADO))
fluxo.sync_light(JOB,DRIVE)
PACOTE=etapa('8-pacote',lambda:core.package(JOB['output']))
etapa('8-backup-pacote',lambda:DRIVE.put(PACOTE,JOB['remote']+'/'+PACOTE.name))
PACOTE_KAGGLE=Path('/kaggle/working/pacote_experimento_24.zip')
shutil.copyfile(PACOTE,PACOTE_KAGGLE)
shutil.copyfile(PAGINA,Path('/kaggle/working/relatorio_24.html'))
print('PACOTE CONCLUÍDO:',PACOTE_KAGGLE)
print('Cópia no Drive:',DRIVE.location(JOB['remote']+'/'+PACOTE.name))
print('Meta 12 classes:',RESULTADO['target_met_primary'],'| Meta secundária 6 classes:',RESULTADO['target_met_secondary'])
display(HTML(PAGINA.read_text(encoding='utf-8')))
display(FileLink('pacote_experimento_24.zip'))
print('Baixe o ZIP acima ou em Output / Files → /kaggle/working ; o pacote completo reúne os resultados da análise.')


## Referências técnicas e limites

- [RF-DETR — treinamento](https://rfdetr.roboflow.com/latest/learn/train/)
- [RF-DETR — API de personalização](https://rfdetr.roboflow.com/latest/learn/train/customization/)
- [RF-DETR — formatos de dataset](https://rfdetr.roboflow.com/latest/learn/train/dataset-formats/)
- [HomeObjects-3K](https://docs.ultralytics.com/datasets/detect/homeobjects-3k)

A implementação fixa RF-DETR **1.11.2**; atualizações da documentação podem descrever outras versões.
O benchmark externo ExDark motivou a hipótese de arquitetura, mas seus percentuais não são uma previsão para Home.
A camada V2 continua heurística e não mede distância real nem comprova segurança de locomoção.

- [Kaggle — notebooks e configurações de sessão](https://www.kaggle.com/docs/notebooks)
